# CNN Encoder — PyTorch
### MedAI Diagnose | CNN + NLP + PEPA

**Architecture:**
```
Image(224,224,3) → EfficientNetB0(frozen, ImageNet)
                 → GlobalAvgPool → Linear(256) → BN → ReLU → Dropout(0.3)
                 → Linear(128) → ReLU
                 → f_img ∈ R^128
```
**Key behaviour:**
- Image = zeros → f_img ≈ 0 → PEPA gate ignores CNN branch
- Image = real → EfficientNetB0 extracts visual features
- Base is FROZEN — only head trains on your data

In [1]:
# Cell 1 — GPU Setup
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
print(f'Device          : {device}')

if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True

PyTorch version : 2.10.0+cu128
CUDA available  : True
Device          : cuda
GPU             : NVIDIA GeForce RTX 5050 Laptop GPU
VRAM            : 8.5 GB


In [2]:
# Cell 2 — CNN Encoder Class

class CNNEncoder(nn.Module):
    """
    EfficientNetB0 image encoder with frozen pretrained base.

    Input  : (batch, 3, 224, 224)  — normalised RGB image [0,1]
    Output : (batch, embed_dim)    — visual embedding f_img

    When input is zero tensor (no image uploaded):
        → EfficientNet outputs near-zero features
        → PEPA gate α → 0 → model runs text-only
        → This is graceful degradation by design
    """
    def __init__(self, embed_dim: int = 128, freeze_base: bool = True):
        super(CNNEncoder, self).__init__()

        # Load EfficientNetB0 pretrained on ImageNet
        base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        # Remove the original classifier head
        # EfficientNetB0 outputs 1280 features after global avg pool
        self.features = base.features          # conv layers
        self.avgpool  = base.avgpool           # global avg pool → (batch, 1280, 1, 1)

        # Freeze the base — use pretrained ImageNet features as-is
        if freeze_base:
            for param in self.features.parameters():
                param.requires_grad = False
            for param in self.avgpool.parameters():
                param.requires_grad = False

        # Trainable embedding head
        self.head = nn.Sequential(
            nn.Flatten(),                      # (batch, 1280)
            nn.Linear(1280, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, embed_dim),
            nn.ReLU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : image tensor (batch, 3, 224, 224) in [0, 1]
        Returns:
            f_img : visual embedding (batch, embed_dim)
        """
        x = self.features(x)
        x = self.avgpool(x)
        x = self.head(x)
        return x


# Build and inspect
cnn = CNNEncoder(embed_dim=128, freeze_base=True).to(device)

total    = sum(p.numel() for p in cnn.parameters())
trainable= sum(p.numel() for p in cnn.parameters() if p.requires_grad)
frozen   = total - trainable

print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}  (head only)')
print(f'Frozen params   : {frozen:,}   (EfficientNetB0 base)')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Aditya Srivastava/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:08<00:00, 2.57MB/s]


Total params    : 4,368,892
Trainable params: 361,344  (head only)
Frozen params   : 4,007,548   (EfficientNetB0 base)


In [3]:
# Cell 3 — Test Real Image vs Zero Image

cnn.eval()
with torch.no_grad():
    # Real image (random pixels simulating actual X-ray)
    real_img  = torch.randn(1, 3, 224, 224).clamp(0, 1).to(device)
    out_real  = cnn(real_img)

    # Zero image (no image uploaded — text-only mode)
    zero_img  = torch.zeros(1, 3, 224, 224).to(device)
    out_zero  = cnn(zero_img)

print(f'Real image → output norm : {out_real.norm().item():.4f}  (should be significant)')
print(f'Zero image → output norm : {out_zero.norm().item():.4f}  (should be small/zero)')
print(f'Output shape             : {out_real.shape}')

print('\n✓ CNN Encoder ready')
print('  When image=zeros → PEPA gate learns to ignore CNN branch')
print('  When image=real  → visual features combined with text')

Real image → output norm : 0.7195  (should be significant)
Zero image → output norm : 0.5057  (should be small/zero)
Output shape             : torch.Size([1, 128])

✓ CNN Encoder ready
  When image=zeros → PEPA gate learns to ignore CNN branch
  When image=real  → visual features combined with text


In [4]:
# Cell 4 — Image Preprocessing Helper
# Used by app.py when patient uploads an image

from PIL import Image
import torchvision.transforms as transforms
import io

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),             # converts to (C, H, W) and scales to [0, 1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],    # ImageNet mean
        std=[0.229, 0.224, 0.225]      # ImageNet std
    )
])

def preprocess_image_bytes(image_bytes: bytes) -> torch.Tensor:
    """
    Convert raw image bytes from upload to model-ready tensor.
    Returns: (1, 3, 224, 224) float32 tensor on CPU
    """
    try:
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB')
        tensor = IMG_TRANSFORM(img).unsqueeze(0)  # add batch dim
        return tensor
    except Exception:
        return torch.zeros(1, 3, 224, 224)

def get_zero_image() -> torch.Tensor:
    """Zero tensor for text-only inference."""
    return torch.zeros(1, 3, 224, 224)

print('✓ Preprocessing helpers defined')
print(f'Transform pipeline: {IMG_TRANSFORM}')

✓ Preprocessing helpers defined
Transform pipeline: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
